# 第4回：データ探偵—分布・欠損・外れ値

**今日の問い：モデルを作る前に、データの怪しいところをどう見つけるか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font="sans-serif")


## TRY：1変数の分布を見る


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="yield_pct", bins=20, ax=axes[0])
axes[0].set_title("収率の分布")
sns.boxplot(data=df, x="reaction_time_h", ax=axes[1])
axes[1].set_title("反応時間：外れ値候補を探す")
plt.tight_layout()


## 2変数の関係とカテゴリ比較


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="temperature_c", y="yield_pct", hue="catalyst", alpha=0.65, ax=axes[0])
axes[0].set_title("温度と収率")
sns.boxplot(data=df, x="catalyst", y="yield_pct", ax=axes[1])
axes[1].set_title("触媒別の収率")
plt.tight_layout()


## TRY：欠損と怪しい値を表で確認


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("欠損数"))
display(df.nlargest(5, "temperature_c")[["sample_id", "temperature_c", "reaction_time_h", "yield_pct"]])


## CHANGE

色分けを`catalyst`から`solvent`へ変えます。見え方が変わった点を1つ共有します。

## 注意

外れ値は入力ミスとは限りません。「誰に確認するか」「残す場合に何が起きるか」まで考えます。
